
# Diffuse ISM Optical Depth (τ_diff)

The diffuse ISM attenuation affects all stellar light (not just young
stars). Higher τ_diff reddens the optical continuum and weakens the
4000 Å break, a signature of aging stellar populations.

.. sphx-glr-precomputed-img:

<img src="file://images/sphx_glr_plot_tau_diff_sweep_001.png" alt="plot_tau_diff_sweep" class="sphx-glr-single-img">


In [ ]:
from pathlib import Path

import jax
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

from tengri import Fixed, Parameters, SEDModel, load_ssp_data
from tengri.analysis.plotting import SWEEP_CMAPS, setup_style, sweep_parameter

setup_style()


def _find_ssp():
    """Find SSP data file in standard locations."""
    name = "ssp_prsc_miles_chabrier_wNE_logGasU-3.0_logGasZ0.0.h5"
    for p in [
        Path("data") / name,
        Path("../data") / name,
        Path("../../data") / name,
        Path("../../../data") / name,
    ]:
        if p.exists():
            return str(p)
    return None


SSP_PATH = _find_ssp()
if SSP_PATH is None:
    raise FileNotFoundError("SSP data not found — skipping example")

ssp = load_ssp_data(SSP_PATH)

# --- Build model: typical star-forming galaxy ---
spec = Parameters(
    sfh_tsnorm_log_peak_sfr=Fixed(1.0),
    sfh_tsnorm_peak_lbt_gyr=Fixed(2.0),  # Peak ~2 Gyr ago
    sfh_tsnorm_width_gyr=Fixed(1.5),
    sfh_tsnorm_skew=Fixed(0.2),
    sfh_tsnorm_trunc=Fixed(3.0),
    met_logzsol=Fixed(-0.3),
    dust_tau_bc=Fixed(0.5),  # Modest birth cloud dust
    dust_tau_diff=Fixed(0.3),  # Will sweep this
    dust_slope=Fixed(-0.7),  # Calzetti-like
    redshift=Fixed(0.1),
)
model = SEDModel(spec, ssp)

# --- Sweep τ_diff ---
values = [0.0, 0.3, 0.7, 1.5, 3.0]

# # The sweep_parameter helper creates a single SEDModel instance and calls
# # model.predict_rest_sed(...) in a loop. JAX JIT compilation is cached
# # automatically via tengri's persistent compilation cache (enabled at
# # import time), so repeated forward model calls reuse the compiled kernel.
fig, ax = sweep_parameter(
    model,
    "dust_tau_diff",
    values,
    cmap=SWEEP_CMAPS["dust"],
    label_fmt=r"$\tau_{{diff}}$ = {:.1f}",
    wave_range=(1000, 10000),
)
ax.set_ylim(0, 1.5e5)
ax.set_title("Diffuse ISM Dust: Impact on Galaxy Attenuation", fontsize=12)
ax.set_ylabel(r"$\lambda F_\lambda$ (normalized at 5500 Å)")
plt.tight_layout()
plt.savefig("plot_tau_diff_sweep.png", dpi=150, bbox_inches="tight")
plt.show()